# Symptom-Driven Multi-Model Comparison: Bayesian RL
We fit 3 models where the session-by-session learning rates are a linear function of a daily symptom cluster.
`alpha(sub, ses) = invlogit( a(sub) + b(sub) * normalized_symptom(sub, ses) )`

1. **Single Alpha**: 1 intercept and slope, 1 global beta.
2. **Valence**: 2 separate intercepts and slopes (PE>0 vs PE<=0), 1 global beta.
3. **Task**: 2 separate intercepts and slopes (Win vs Lose domain), 2 global betas (Win vs Lose).

Models are executed using the rapid `numpyro` backend. The winning model will plot Q-values over time, alongside the posterior probability distribution for the group-level slope (`mu_b`).


In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import pandas as pd
import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az
import warnings

# Import custom session-varying learning functions
from learning_functions_ses import update_m1_ses, update_m2_ses, update_m3_ses

warnings.filterwarnings('ignore')


In [ ]:
# ==========================================
# SYMPTOM CONFIGURATION
# ==========================================
SYMPTOM_CHOICES = [
    're_exp_int',               # 0
    're_exp_ext',               # 1
    'avoidance',                # 2
    'Negative_affect',          # 3
    'Emotional_numbing',        # 4
    'Externalizing_behaviors',  # 5
    'Anxious_arousal',          # 6
    'Dysphoric_arousal',        # 7
    'Total_Symptoms'            # 8
]

# CHANGE THIS INDEX (0 to 8) TO SELECT THE SYMPTOM TO ANALYZE
SELECTED_SYMPTOM_IDX = 8
selected_symptom = SYMPTOM_CHOICES[SELECTED_SYMPTOM_IDX]

print(f"Selected Symptom: {selected_symptom}")


In [ ]:
file_path = '../data/PSUB2_RL_data_and_questionnaire_all_subs_4.21.26.xlsx'
df = pd.read_excel(file_path)
df = df.dropna(subset=['ChoiceKey']).copy()

stim_map = {'W1': 0, 'W2': 1, 'L1': 2, 'L2': 3}
df['LeftID_int'] = df['LeftID'].map(stim_map)
df['RightID_int'] = df['RightID'].map(stim_map)
df['ChosenID_int'] = df['ChosenID'].map(stim_map)
df['Choice'] = df['ChoiceKey'] - 1
df['RewardScaled'] = df['TrialPoints'] / 30.0
df['is_win_trial'] = (df['TrialType'] == 'W').astype(int)

# Sort by Sub, Ses, Trial
df = df.sort_values(['Sub', 'Ses', 'Trial']).reset_index(drop=True)

# Create SubIdx (0 to n_sub-1)
unique_subs = df['Sub'].unique()
sub_idx_map = {sub: i for i, sub in enumerate(unique_subs)}
df['SubIdx'] = df['Sub'].map(sub_idx_map)
n_sub = len(unique_subs)

# Create SesIdx (0 to n_ses-1)
unique_ses = np.sort(df['Ses'].unique())
ses_idx_map = {ses: i for i, ses in enumerate(unique_ses)}
df['SesIdx'] = df['Ses'].map(ses_idx_map)
n_ses = len(unique_ses)

df['is_first'] = 0
for _, group in df.groupby(['Sub', 'Ses']):
    df.loc[group.index[0], 'is_first'] = 1

# Merge Symptom Data
cluster_sizes = {
    're_exp_int': 3,
    're_exp_ext': 2,
    'avoidance': 2,
    'Negative_affect': 4,
    'Emotional_numbing': 3,
    'Externalizing_behaviors': 2,
    'Anxious_arousal': 2,
    'Dysphoric_arousal': 2,
    'Total_Symptoms': 20
}
max_score = cluster_sizes[selected_symptom] * 4

df_symp = pd.read_csv('../data/symptoms_table.csv')
df = pd.merge(df, df_symp[['Sub', 'Ses', selected_symptom]], on=['Sub', 'Ses'], how='left')

# Forward fill or drop NAs if symptom data is missing for some sessions
df[selected_symptom] = df[selected_symptom].fillna(0) # Assuming 0 if missing, adjust if needed
df['Normalized_Symptom'] = df[selected_symptom] / max_score

# Create (n_sub, n_ses) constant matrix for fast lookup
symp_matrix = np.zeros((n_sub, n_ses))
for sub, group in df.groupby('SubIdx'):
    for ses, ses_group in group.groupby('SesIdx'):
        symp_matrix[sub, ses] = ses_group['Normalized_Symptom'].iloc[0]

print(f"Symptom scaling max score: {max_score}")
print(f"Total subjects: {n_sub}")
print(f"Total sessions per subject (max): {n_ses}")
print(f"Total trials: {len(df)}")


In [ ]:
def make_symptom_alpha(name, n_sub, symp_matrix):
    """Creates a linear equation for alpha constrained by invlogit"""
    mu_a = pm.Normal(f'{name}_mu_a', mu=0, sigma=1)
    sigma_a = pm.HalfNormal(f'{name}_sigma_a', sigma=1)
    a = pm.Normal(f'{name}_a', mu=mu_a, sigma=sigma_a, shape=n_sub)
    
    mu_b = pm.Normal(f'{name}_mu_b', mu=0, sigma=1)
    sigma_b = pm.HalfNormal(f'{name}_sigma_b', sigma=1)
    b = pm.Normal(f'{name}_b', mu=mu_b, sigma=sigma_b, shape=n_sub)
    
    # Linear equation: a[sub] + b[sub] * symp_matrix[sub, ses]
    alpha_logit = a[:, None] + b[:, None] * symp_matrix
    return pm.Deterministic(name, pm.math.invlogit(alpha_logit))

def make_hierarchical_beta(name, n_sub):
    mu = pm.Normal(f'{name}_mu', mu=0, sigma=2.0)
    sigma = pm.HalfNormal(f'{name}_sigma', sigma=1.0)
    raw = pm.Normal(f'{name}_raw', mu=0, sigma=1, shape=n_sub)
    return pm.Deterministic(name, pm.math.exp(mu + raw * sigma))

def build_model_1(df, n_sub, symp_matrix):
    with pm.Model() as m1:
        alpha = make_symptom_alpha('alpha', n_sub, symp_matrix)
        beta = make_hierarchical_beta('beta', n_sub)
        
        Q_init = pt.zeros((n_sub, 4))
        outputs_info = [Q_init, None, None, None]
        
        results, updates = pytensor.scan(
            fn=update_m1_ses,
            sequences=[
                pt.as_tensor_variable(df['LeftID_int'].values),
                pt.as_tensor_variable(df['RightID_int'].values),
                pt.as_tensor_variable(df['ChosenID_int'].values),
                pt.as_tensor_variable(df['RewardScaled'].values),
                pt.as_tensor_variable(df['is_first'].values),
                pt.as_tensor_variable(df['SubIdx'].values),
                pt.as_tensor_variable(df['SesIdx'].values)
            ],
            outputs_info=outputs_info,
            non_sequences=[alpha],
            strict=True
        )
        
        q_left_seq = results[1]
        q_right_seq = results[2]
        Q_reset_seq = results[3]
        pm.Deterministic('Q_values_pre_update', Q_reset_seq)
        
        sub_idx_seq = pt.as_tensor_variable(df['SubIdx'].values)
        p_right = pm.math.sigmoid(beta[sub_idx_seq] * (q_right_seq - q_left_seq))
        pm.Deterministic('p_right', p_right)
        
        pm.Bernoulli('choice_obs', p=p_right, observed=df['Choice'].values)
    return m1

def build_model_2(df, n_sub, symp_matrix):
    with pm.Model() as m2:
        alpha_pos = make_symptom_alpha('alpha_pos', n_sub, symp_matrix)
        alpha_neg = make_symptom_alpha('alpha_neg', n_sub, symp_matrix)
        beta = make_hierarchical_beta('beta', n_sub)
        
        Q_init = pt.zeros((n_sub, 4))
        outputs_info = [Q_init, None, None, None]
        
        results, updates = pytensor.scan(
            fn=update_m2_ses,
            sequences=[
                pt.as_tensor_variable(df['LeftID_int'].values),
                pt.as_tensor_variable(df['RightID_int'].values),
                pt.as_tensor_variable(df['ChosenID_int'].values),
                pt.as_tensor_variable(df['RewardScaled'].values),
                pt.as_tensor_variable(df['is_first'].values),
                pt.as_tensor_variable(df['SubIdx'].values),
                pt.as_tensor_variable(df['SesIdx'].values)
            ],
            outputs_info=outputs_info,
            non_sequences=[alpha_pos, alpha_neg],
            strict=True
        )
        
        q_left_seq = results[1]
        q_right_seq = results[2]
        Q_reset_seq = results[3]
        pm.Deterministic('Q_values_pre_update', Q_reset_seq)
        
        sub_idx_seq = pt.as_tensor_variable(df['SubIdx'].values)
        p_right = pm.math.sigmoid(beta[sub_idx_seq] * (q_right_seq - q_left_seq))
        pm.Deterministic('p_right', p_right)
        
        pm.Bernoulli('choice_obs', p=p_right, observed=df['Choice'].values)
    return m2

def build_model_3(df, n_sub, symp_matrix):
    with pm.Model() as m3:
        alpha_win = make_symptom_alpha('alpha_win', n_sub, symp_matrix)
        alpha_lose = make_symptom_alpha('alpha_lose', n_sub, symp_matrix)
        beta_win = make_hierarchical_beta('beta_win', n_sub)
        beta_lose = make_hierarchical_beta('beta_lose', n_sub)
        
        Q_init = pt.zeros((n_sub, 4))
        outputs_info = [Q_init, None, None, None]
        
        results, updates = pytensor.scan(
            fn=update_m3_ses,
            sequences=[
                pt.as_tensor_variable(df['LeftID_int'].values),
                pt.as_tensor_variable(df['RightID_int'].values),
                pt.as_tensor_variable(df['ChosenID_int'].values),
                pt.as_tensor_variable(df['RewardScaled'].values),
                pt.as_tensor_variable(df['is_first'].values),
                pt.as_tensor_variable(df['is_win_trial'].values),
                pt.as_tensor_variable(df['SubIdx'].values),
                pt.as_tensor_variable(df['SesIdx'].values)
            ],
            outputs_info=outputs_info,
            non_sequences=[alpha_win, alpha_lose],
            strict=True
        )
        
        q_left_seq = results[1]
        q_right_seq = results[2]
        Q_reset_seq = results[3]
        pm.Deterministic('Q_values_pre_update', Q_reset_seq)
        
        sub_idx_seq = pt.as_tensor_variable(df['SubIdx'].values)
        is_win_seq = pt.as_tensor_variable(df['is_win_trial'].values)
        
        beta_seq = pt.switch(is_win_seq, beta_win[sub_idx_seq], beta_lose[sub_idx_seq])
        p_right = pm.math.sigmoid(beta_seq * (q_right_seq - q_left_seq))
        pm.Deterministic('p_right', p_right)
        
        pm.Bernoulli('choice_obs', p=p_right, observed=df['Choice'].values)
    return m3


In [ ]:
print("--- Building and Sampling Symptom Model 1 ---")
m1 = build_model_1(df, n_sub, symp_matrix)
with m1:
    idata_1 = pm.sample(nuts_sampler="numpyro", progressbar=False, idata_kwargs={'log_likelihood': True})

print("--- Building and Sampling Symptom Model 2 ---")
m2 = build_model_2(df, n_sub, symp_matrix)
with m2:
    idata_2 = pm.sample(nuts_sampler="numpyro", progressbar=False, idata_kwargs={'log_likelihood': True})

print("--- Building and Sampling Symptom Model 3 ---")
m3 = build_model_3(df, n_sub, symp_matrix)
with m3:
    idata_3 = pm.sample(nuts_sampler="numpyro", progressbar=False, idata_kwargs={'log_likelihood': True})

comp_dict = {
    "1. single alpha (symptom)": idata_1,
    "2. valence (symptom)": idata_2,
    "3. task (symptom)": idata_3
}
comp = az.compare(comp_dict)
print("\nSymptom Model Comparison (LOO):")
display(comp)


In [ ]:
def get_p_opt1(row, opt1_id):
    if row['RightID'] == opt1_id:
        return row['P_Right']
    elif row['LeftID'] == opt1_id:
        return 1.0 - row['P_Right']
    return np.nan

def plot_slopes(idata, model_name):
    # Find all variables ending in '_mu_b' (the group-level slope)
    b_vars = [var for var in idata.posterior.data_vars if var.endswith('_mu_b')]
    
    for b_var in b_vars:
        post = idata.posterior[b_var].values.flatten()
        prob_above_0 = np.mean(post > 0)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        az.plot_posterior(post, point_estimate='mean', hdi_prob=0.95, ax=ax)
        ax.set_title(f"{model_name} - {b_var}\n{prob_above_0*100:.1f}% of distribution > 0")
        plt.tight_layout()
        plt.show()

def plot_winning_model(df, idata, model_name, n_sub):
    Q_mean = idata.posterior['Q_values_pre_update'].mean(dim=['chain', 'draw']).values
    p_right_mean = idata.posterior['p_right'].mean(dim=['chain', 'draw']).values
    
    df = df.copy()
    df['P_Right'] = p_right_mean
    df['P_W1'] = df.apply(lambda r: get_p_opt1(r, 'W1') if r['TrialType'] == 'W' else np.nan, axis=1)
    df['P_L1'] = df.apply(lambda r: get_p_opt1(r, 'L1') if r['TrialType'] == 'L' else np.nan, axis=1)
    
    unique_subs = df['Sub'].unique()
    
    for sub in unique_subs:
        df_sub = df[df['Sub'] == sub]
        sub_idx = df_sub['SubIdx'].iloc[0]
        sessions = df_sub['Ses'].unique()
        
        for ses in sessions:
            df_ses = df_sub[df_sub['Ses'] == ses].copy()
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
            fig.suptitle(f"Subject {sub} - Session {ses} | Normalized {selected_symptom}: {df_ses['Normalized_Symptom'].iloc[0]:.2f}", fontsize=14)
            
            # Win Domain
            ax_w = axes[0]
            df_w = df_ses[df_ses['TrialType'] == 'W']
            if not df_w.empty:
                x_w = df_w['IdxWithinType'].values
                choices_w = (df_w['ChosenID'] == 'W1').astype(int).values
                ax_w.scatter(x_w, choices_w, color='black', alpha=0.6, label='Choice (1=W1, 0=W2)')
                ax_w.plot(x_w, df_w['P_W1'].values, color='green', label='P(Choose W1)', lw=2)
                
                ax_w.set_title("Win Domain (W1 vs W2)")
                ax_w.set_xlabel("IdxWithinType")
                ax_w.set_yticks([0, 1])
                ax_w.set_yticklabels(['W2', 'W1'])
                ax_w.set_ylim(-0.1, 1.1)
                
                ax_qw = ax_w.twinx()
                ax_qw.plot(x_w, Q_mean[df_w.index, sub_idx, 0], color='blue', linestyle='--', label='Q(W1)')
                ax_qw.plot(x_w, Q_mean[df_w.index, sub_idx, 1], color='orange', linestyle='--', label='Q(W2)')
                ax_qw.set_ylabel("Q-value (Scaled)")
                
                lines, labels = ax_w.get_legend_handles_labels()
                lines2, labels2 = ax_qw.get_legend_handles_labels()
                ax_w.legend(lines + lines2, labels + labels2, loc='center right')
                
            # Lose Domain
            ax_l = axes[1]
            df_l = df_ses[df_ses['TrialType'] == 'L']
            if not df_l.empty:
                x_l = df_l['IdxWithinType'].values
                choices_l = (df_l['ChosenID'] == 'L1').astype(int).values
                ax_l.scatter(x_l, choices_l, color='black', alpha=0.6, label='Choice (1=L1, 0=L2)')
                ax_l.plot(x_l, df_l['P_L1'].values, color='red', label='P(Choose L1)', lw=2)
                
                ax_l.set_title("Loss Domain (L1 vs L2)")
                ax_l.set_xlabel("IdxWithinType")
                ax_l.set_yticks([0, 1])
                ax_l.set_yticklabels(['L2', 'L1'])
                ax_l.set_ylim(-0.1, 1.1)
                
                ax_ql = ax_l.twinx()
                ax_ql.plot(x_l, Q_mean[df_l.index, sub_idx, 2], color='purple', linestyle='--', label='Q(L1)')
                ax_ql.plot(x_l, Q_mean[df_l.index, sub_idx, 3], color='brown', linestyle='--', label='Q(L2)')
                ax_ql.set_ylabel("Q-value (Scaled)")
                
                lines, labels = ax_l.get_legend_handles_labels()
                lines2, labels2 = ax_ql.get_legend_handles_labels()
                ax_l.legend(lines + lines2, labels + labels2, loc='center right')
                
            plt.tight_layout()
            plt.show()

winning_model_name = comp.index[1]
winning_idata = comp_dict[winning_model_name]
print(f"\n{'='*50}\nPlotting Posterior Slopes for {winning_model_name}\n{'='*50}")
plot_slopes(winning_idata, winning_model_name)

print(f"\n{'='*50}\nPlotting Behavioral Results for {winning_model_name}\n{'='*50}")
plot_winning_model(df, winning_idata, winning_model_name, n_sub)
